# Lab 03 — Certified robustness

In Lab 02, an attack searched for an input that fools a classifier. Here we ask a different question: **can we guarantee that every input in a small neighborhood is classified correctly?**

This practical adapts the [auto-LiRPA quick-start tutorial](http://PaperCode.cc/AutoLiRPA-Demo). We keep its small pretrained ResNet-18 and CIFAR-10 example, then compare bounds, turn them into certificates, and connect the results to an attack.

By the end, you should be able to:

- Define an $L_\infty$ perturbation in pixel space.
- Run interval bound propagation (IBP) and CROWN with auto-LiRPA.
- Read a certificate and distinguish an inconclusive bound from an attack success.

**Suggested duration:** 60–90 minutes. All core code is provided. Run it in order and answer the three short exercises. The final extension is optional.

## 1. Imports and setup

Use this lab's environment: run `uv sync` in `src/03_certified_robustness`, then select its `.venv` notebook kernel. Follow [README.md](README.md) to prepare your named submission copy. Downloads are cached in `data/`.

The lab runs on CPU; CUDA is used if available. We do not install or uninstall packages inside the notebook.

In [ ]:
import sys
import warnings
from time import perf_counter
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.nn import functional as F
from torchvision import datasets, transforms
from IPython.display import display
from auto_LiRPA import BoundedModule, BoundedTensor, PerturbationLpNorm

from lab_utils import CIFAR_STD, load_demo_model

# Matrix mode is intentional for this tiny network. Hide its repeated size notices.
warnings.filterwarnings(
    "ignore", message=r"Creating an identity matrix with size .*",
    category=UserWarning, module=r"auto_LiRPA\.bound_general",
)
# Upstream sets a global warning filter; keep third-party deprecations out of the lab output.
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"matplotlib\._mathtext|pyparsing.*")

assert sys.version_info[:2] == (3, 11), "Select Lab 03's Python 3.11 .venv kernel."
SEED = 3
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}; torch: {torch.__version__}; auto-LiRPA: {version('auto-lirpa')}")

## 2. Model and data

We use the tutorial's **narrow ResNet-18** (starting with just two channels). Its architecture is in [lab_utils.py](lab_utils.py). It has residual connections and batch normalization, but is small enough for this practical. The checkpoint was naturally trained: expect certificates only for small perturbations.

The model accepts pixels in **[0, 1]** and normalizes them internally. This puts preprocessing inside both the attack and the verified computation. We keep batch normalization in evaluation mode throughout the lab.

We study one fixed test image. Its label and prediction below are computed from the current run, rather than assumed in the text. This example does not measure performance on the full test set.

In [ ]:
model = load_demo_model(device)
model.requires_grad_(False)  # Evaluation and PGD need input gradients only.
test_data = datasets.CIFAR10(
    "data", train=False, download=True, transform=transforms.ToTensor()
)
INDEX = 123
pixels, label = test_data[INDEX]
image = pixels.unsqueeze(0).to(device)
labels = torch.tensor([label], device=device)
classes = test_data.classes

with torch.no_grad():
    logits = model(image)
predicted = logits.argmax(dim=1)
clean_correct = predicted.eq(labels)
print(f"Image {INDEX}: true label = {classes[label]}, prediction = {classes[predicted.item()]}")
print(f"Clean classification correct: {clean_correct.item()}")
plt.figure(figsize=(3, 3))
plt.imshow(pixels.permute(1, 2, 0))
plt.title(f"True: {classes[label]} / predicted: {classes[predicted.item()]}")
plt.axis("off")
plt.show()

## 3. Define the threat model

For an image $x$, our allowed inputs are

$$\mathcal{S}(x,\epsilon)=\{x' : \|x'-x\|_\infty\leq\epsilon,\quad 0\leq x'\leq1\}.$$

Every color channel of every pixel can change by at most $\epsilon$. We clip the interval to valid pixels. Perturbations are continuous; they need not be multiples of $1/255$.

**Units matter:** an epsilon of $1/255$ means one intensity level on the usual 0–255 scale. After normalization $(x-\mu_c)/\sigma_c$, the same perturbation is $\epsilon/\sigma_c$ in channel $c$. Applying a scalar epsilon directly to normalized inputs would define a different threat model.

auto-LiRPA needs three steps: wrap the model, wrap a perturbed input, and call `compute_bounds`.

In [ ]:
# Step 1: trace the model once. Its input is raw pixels.
bounded_model = BoundedModule(
    model, torch.zeros_like(image),
    # Keep parameter gradients available in eval mode for the optional extension.
    bound_opts={"conv_mode": "matrix", "param": {"auto_requires_grad": False}}, device=device,
)
bounded_model.eval()
with torch.no_grad():
    torch.testing.assert_close(bounded_model(image), logits, atol=1e-5, rtol=1e-4)

# Step 2: describe the allowed box, including the [0, 1] pixel constraints.
def bounded_input(x, epsilon):
    if epsilon < 0:
        raise ValueError("epsilon must be nonnegative")
    x_L = (x - epsilon).clamp(0, 1)
    x_U = (x + epsilon).clamp(0, 1)
    perturbation = PerturbationLpNorm(
        norm=np.inf, eps=epsilon, x_L=x_L, x_U=x_U,
    )
    return BoundedTensor(x, perturbation)

EPSILON = 0.1 / 255  # Small starting radius for this naturally trained model.
bounded_image = bounded_input(image, EPSILON)
print(f"Pixel-space epsilon: {EPSILON:.6f} ({EPSILON * 255:g}/255)")
print("Equivalent normalized channel radii:", np.array(CIFAR_STD) ** -1 * EPSILON)

### Exercise 1 — Epsilon and preprocessing (about 5 minutes)

1. If `epsilon = 2 / 255`, what is the largest allowed change in a pixel on the 0–255 scale?
2. Why do we include normalization inside the model instead of perturbing already normalized inputs?
3. A pixel has value `0.999` and epsilon is `2 / 255`. What are its lower and upper allowed values?

**Your answers:**

_Write your answers here._

## 4. Bound the output logits

For each output logit $f_j$, auto-LiRPA returns bounds valid for every input in the box:

$$\ell_j\leq f_j(x')\leq u_j \quad\text{for all }x'\in\mathcal{S}(x,\epsilon).$$

We compare two methods:

- **IBP** propagates intervals through the network. It is usually fast, but dependencies between quantities can be lost.
- **CROWN** propagates linear bounds backward through the graph. It often gives tighter bounds, at a higher computational cost.

These methods overapproximate the possible outputs. A bound endpoint need not be attainable by an actual perturbed input.

In [ ]:
# Step 3: compute the bounds. Timing includes device synchronization on CUDA.
def compute_bounds(x, epsilon, method, C=None, lower_only=False):
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = perf_counter()
    with torch.no_grad():
        lb, ub = bounded_model.compute_bounds(
            x=(bounded_input(x, epsilon),), method=method, C=C,
            bound_upper=not lower_only,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = perf_counter() - start
    return lb.detach(), None if ub is None else ub.detach(), elapsed

METHODS = ("IBP", "CROWN")
logit_bounds = {}
for method in METHODS:
    lb, ub, seconds = compute_bounds(image, EPSILON, method)
    assert torch.all(lb <= ub + 1e-5)
    assert torch.all(logits >= lb - 1e-4) and torch.all(logits <= ub + 1e-4)
    logit_bounds[method] = (lb, ub)
    print(f"{method}: {seconds:.3f} s; mean interval width = {(ub - lb).mean().item():.4f}")

logit_table = pd.DataFrame({"class": classes, "clean logit": logits[0].cpu().numpy()})
for method, (lb, ub) in logit_bounds.items():
    logit_table[f"{method} lower"] = lb[0].cpu().numpy()
    logit_table[f"{method} upper"] = ub[0].cpu().numpy()
display(logit_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
positions = np.arange(len(classes))
for ax, method in zip(axes, METHODS):
    lb, ub = (v[0].cpu().numpy() for v in logit_bounds[method])
    centers = (lb + ub) / 2
    ax.errorbar(positions, centers, yerr=(ub - lb) / 2,
                fmt="o", capsize=3, label="Bounds")
    ax.scatter(positions, logits[0].cpu().numpy(), c="black", marker="x", label="Clean logit")
    ax.set_xticks(positions, classes, rotation=60)
    ax.set_ylabel("Logit (each panel has its own scale)")
    ax.set_title(f"{method} at epsilon = {EPSILON * 255:g}/255")
    ax.legend()
fig.tight_layout()
plt.show()

## 5. Turn bounds into a certificate

The classifier is correct throughout the neighborhood if the true class $y$ always beats every competing class:

$$f_y(x')-f_j(x')>0 \quad\text{for every }j\ne y,\;x'\in\mathcal{S}(x,\epsilon).$$

One sufficient condition is $\ell_y-\max_{j\ne y}u_j>0$. We can also ask auto-LiRPA to **bound the differences directly**, which can avoid losing dependencies between logits.

The specification matrix `C` has one row $e_y-e_j$ per competitor. Multiplying it by the logit vector gives the nine margins $f_y-f_j$. We certify only if every margin lower bound is positive. The small tolerance below avoids reporting a nearly zero floating-point bound as a certificate.

We certify **correctness relative to the true label**. If an image is already misclassified, preserving its predicted class would be a different property.

In [ ]:
def margin_specification(target_labels, num_classes=10):
    # C has shape (batch size, number of competitors, number of classes).
    batch = target_labels.numel()
    all_classes = torch.arange(num_classes, device=target_labels.device)
    competitors = all_classes.expand(batch, -1)
    competitors = competitors[competitors != target_labels[:, None]].view(batch, num_classes - 1)
    eye = torch.eye(num_classes, device=target_labels.device)
    C = eye[target_labels][:, None, :] - eye[competitors]
    return C, competitors

C, competitors = margin_specification(labels)
CERT_TOL = 1e-6
certificate_rows = []
margin_bounds = {}
for method in METHODS:
    lb_logits, ub_logits = logit_bounds[method]
    separate = lb_logits.gather(1, labels[:, None]) - ub_logits.gather(1, competitors)
    lb_margin, _, seconds = compute_bounds(image, EPSILON, method, C=C, lower_only=True)
    margin_bounds[method] = lb_margin
    certificate_rows.append({
        "method": method,
        "separate-logit minimum": separate.min().item(),
        "direct-margin minimum": lb_margin.min().item(),
        "certified correct": bool(clean_correct.item() and (lb_margin > CERT_TOL).all()),
        "seconds": seconds,
    })
certificates = pd.DataFrame(certificate_rows)
display(certificates.round(6))
display(pd.DataFrame({
    "competitor": [classes[j] for j in competitors[0].tolist()],
    **{method: bounds[0].cpu().numpy() for method, bounds in margin_bounds.items()},
}).round(4))

### Exercise 2 — Read the certificate (about 10 minutes)

1. At the starting epsilon, which methods certify this image? Quote their minimum margin lower bounds.
2. Which method has tighter logit intervals in this run? Which takes longer? Small timings can vary between runs.
3. If one margin lower bound is negative, have we proved that an adversarial example exists? Explain.

**Your answers:**

_Write your answers here._

## 6. Compare certification with an attack

We reuse the idea of projected gradient descent (PGD) from Lab 02. The implementation is supplied: it maximizes cross-entropy, projects each step into **the same clipped pixel box**, and uses random restarts. It retains any misclassification found along the way.

- A **positive certificate** covers every input in the box.
- A **successful attack** provides a concrete counterexample.
- If certification fails and PGD finds nothing, the result is **unresolved**. Attack failure alone is not a proof of robustness.

The code checks that attacks stay inside the threat model and that a successful attack never contradicts a reported certificate.

In [ ]:
def pgd_attack(x, y, epsilon, steps=20, restarts=2):
    if epsilon == 0:
        return x.detach().clone()
    x_L, x_U = (x - epsilon).clamp(0, 1), (x + epsilon).clamp(0, 1)
    step_size = epsilon / 4
    best = x.detach().clone()
    with torch.no_grad():
        best_logits = model(best)
        best_loss = F.cross_entropy(best_logits, y, reduction="none")
        best_wrong = best_logits.argmax(1).ne(y)
    for _ in range(restarts):
        candidate = x_L + torch.rand_like(x) * (x_U - x_L)
        for step in range(steps + 1):
            candidate = candidate.detach().requires_grad_(True)
            output = model(candidate)
            losses = F.cross_entropy(output, y, reduction="none")
            wrong = output.argmax(1).ne(y)
            # Preserve a counterexample even if a later step has greater loss.
            improve = (wrong & ~best_wrong) | ((wrong == best_wrong) & (losses.detach() > best_loss))
            best[improve] = candidate.detach()[improve]
            best_loss[improve] = losses.detach()[improve]
            best_wrong[improve] = wrong.detach()[improve]
            if step < steps:
                gradient = torch.autograd.grad(losses.sum(), candidate)[0]
                candidate = candidate.detach() + step_size * gradient.sign()
                candidate = torch.maximum(torch.minimum(candidate, x_U), x_L)
    return best.detach()

# Verify our construction at epsilon = 0: bounds should recover the clean margins.
zero_lb, _, _ = compute_bounds(image, 0, "CROWN", C=C, lower_only=True)
clean_margins = logits.gather(1, labels[:, None]) - logits.gather(1, competitors)
torch.testing.assert_close(zero_lb, clean_margins, atol=1e-4, rtol=1e-4)

### Increase epsilon

We use the same image, label, model and pixel constraints for every row. The radius is shown in units of `1 / 255`. The results concern this one image; the proportion of radii certified is not a certified accuracy.

In [ ]:
EPSILON_LEVELS = [0, 0.1, 0.5, 1, 4]
rows = []
attacked_images = {}
torch.manual_seed(SEED)
for level in EPSILON_LEVELS:
    epsilon = level / 255
    attacked = pgd_attack(image, labels, epsilon)
    attacked_images[level] = attacked
    with torch.no_grad():
        attack_prediction = model(attacked).argmax(1)
    found_error = bool(attack_prediction.ne(labels).item())
    assert (attacked - image).abs().max().item() <= epsilon + 1e-6
    assert attacked.min().item() >= 0 and attacked.max().item() <= 1
    for method in METHODS:
        lb_margin, _, seconds = compute_bounds(image, epsilon, method, C=C, lower_only=True)
        certified = bool(clean_correct.item() and (lb_margin > CERT_TOL).all())
        assert not (certified and found_error), "Certificate and attack disagree: check the threat model."
        status = ("clean error" if not clean_correct.item() else
                  "certified" if certified else
                  "attack found" if found_error else "unresolved")
        rows.append({"epsilon (1/255)": level, "method": method,
                     "minimum margin lower bound": lb_margin.min().item(),
                     "certified correct": certified, "PGD found error": found_error,
                     "status": status, "seconds": seconds})
results = pd.DataFrame(rows)
display(results.round(6))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for method in METHODS:
    subset = results[results["method"] == method]
    ax.plot(subset["epsilon (1/255)"], subset["minimum margin lower bound"], "o-", label=method)
ax.axhline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Epsilon (units of 1/255)")
ax.set_yscale("symlog", linthresh=10)  # Show small margins and very loose bounds together.
ax.set_ylabel("Minimum margin bound (symlog)")
ax.set_title("A positive lower bound certifies correctness")
ax.legend()
fig.tight_layout()
plt.show()

largest_level = max(EPSILON_LEVELS)
adversarial = attacked_images[largest_level]
with torch.no_grad():
    adv_label = model(adversarial).argmax(1).item()
delta = (adversarial - image)[0].cpu()
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(pixels.permute(1, 2, 0))
axes[0].set_title(f"Clean: {classes[predicted.item()]}")
axes[1].imshow(adversarial[0].cpu().permute(1, 2, 0))
axes[1].set_title(f"PGD: {classes[adv_label]}")
# Color encodes signed change: gray means zero; the difference is magnified.
axes[2].imshow((0.5 + delta / (2 * largest_level / 255)).clamp(0, 1).permute(1, 2, 0))
axes[2].set_title("Magnified difference")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
plt.show()
print(f"PGD radius: {largest_level}/255; actual maximum change: {delta.abs().max().item():.6f}")

### Exercise 3 — A small experiment (about 15 minutes)

1. From the table, report the largest **tested** epsilon certified by each method. This is not necessarily the exact maximum robust radius.
2. Find a row with status `unresolved` if there is one. Explain what we know and what remains unknown. If there is none, describe what that status would mean.
3. Change `EPSILON_LEVELS` to include `2` and `8`, then rerun the sweep and its plots. Does PGD find an error at either radius? Does either bound certify it? Report the results without assuming the outcome in advance.

**Your answers:**

_Write your answers here._

## 7. Optional extension — Differentiable bounds

The core practical ends above. If you have time, set `RUN_EXTENSION = True` and run the cell below.

The IBP bounds are differentiable with respect to model parameters. Certified training can use such gradients to improve bounds. We only compute a gradient here; we do not update the weights. Evaluation mode keeps batch-normalization statistics fixed and still permits gradients.

In [ ]:
RUN_EXTENSION = False
if RUN_EXTENSION:
    bounded_model.eval()
    bounded_model.requires_grad_(True)
    bounded_model.zero_grad(set_to_none=True)
    differentiable_lb, _ = bounded_model.compute_bounds(
        x=(bounded_input(image, EPSILON),), method="IBP", C=C, bound_upper=False,
    )
    objective = -differentiable_lb.mean()  # Minimizing this would increase average margin bounds.
    objective.backward()
    gradient_norms = [p.grad.norm().item() for p in bounded_model.parameters() if p.grad is not None]
    assert gradient_norms and np.isfinite(gradient_norms).all()
    print(f"Gradient norms: {len(gradient_norms)} parameters; largest norm = {max(gradient_norms):.6f}")
    bounded_model.zero_grad(set_to_none=True)
    bounded_model.requires_grad_(False)
else:
    print("Optional extension skipped.")

## Submission and further reading

Save the notebook with your name in its filename, including your three answers and the outputs of your experiments. Submit it following the course instructions. Keep the main demonstration's outputs so your answers can be checked against your run.

To go further, repeat the experiment for another `INDEX`. If the clean prediction is incorrect, explain why a certificate of correct classification is impossible for that neighborhood. For certified accuracy on multiple images, count certified **correct** images and divide by the total number evaluated, including clean errors.

References:

- [Original auto-LiRPA quick-start tutorial](http://PaperCode.cc/AutoLiRPA-Demo).
- [Official verification example](https://github.com/Verified-Intelligence/auto_LiRPA/blob/a050a3d61f4cb68b108fa7f07dcb3e4d7ef304df/examples/vision/simple_verification.py).
- [auto-LiRPA examples and certified training](https://auto-lirpa.readthedocs.io/en/latest/examples.html).
- [CROWN: Efficient Neural Network Robustness Certification with General Activation Functions](https://arxiv.org/abs/1811.00866).

The architecture attribution is recorded in [THIRD_PARTY_NOTICES.md](THIRD_PARTY_NOTICES.md).